# 3 · Fehler und Konvergenz

Eine numerische Rechnung ist erst überzeugend, wenn der Fehler bei Verfeinerung systematisch abnimmt. Wir messen am Endzeitpunkt den diskreten $L^2$-Fehler

$$\lVert u_h-u\rVert_{L^2} \approx \left(\sum_j |u_h(x_j)-u(x_j)|^2h\right)^{1/2}.$$

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project = Path.cwd()
if not (project / 'galerkin_heat.py').exists() and (project.parent / 'galerkin_heat.py').exists():
    project = project.parent
if not (project / 'galerkin_heat.py').exists():
    from urllib.request import urlretrieve
    source = 'https://raw.githubusercontent.com/karkessler/dhbw-mathe3/main/galerkin/galerkin_heat.py'
    urlretrieve(source, project / 'galerkin_heat.py')
sys.path.insert(0, str(project))

from galerkin_heat import l2_error, solve_heat_equation


In [ ]:
element_counts = np.array([8, 12, 16, 24, 32, 48])
spatial_errors = []
for elements in element_counts:
    # Sehr kleiner Zeitschritt, damit vor allem der Ortsfehler sichtbar wird.
    result = solve_heat_equation(elements=elements, dt=0.0001, final_time=0.05)
    spatial_errors.append(l2_error(result))

spatial_errors = np.array(spatial_errors)
for elements, error in zip(element_counts, spatial_errors):
    print(f'{elements:3d} Elemente: L2-Fehler = {error:.4e}')


In [ ]:
plt.figure(figsize=(7, 4))
plt.loglog(1.0 / element_counts, spatial_errors, 'o-', label='gemessener Fehler')
reference = spatial_errors[-1] * (element_counts[-1] / element_counts) ** 2
plt.loglog(1.0 / element_counts, reference, '--', label=r'Referenz $O(h^2)$')
plt.xlabel('Gitterweite h')
plt.ylabel('L2-Fehler')
plt.title('Räumliche Konvergenz')
plt.grid(which='both', alpha=0.3)
plt.legend()
plt.show()


## Zeitdiskretisierung

Crank–Nicolson besitzt für glatte Lösungen die Fehlerordnung zwei. Für diesen Versuch verwenden wir ein feines Ortsgitter und variieren nur den Zeitschritt.

In [ ]:
time_steps = np.array([0.02, 0.01, 0.005, 0.0025])
time_errors = []
for dt in time_steps:
    result = solve_heat_equation(elements=100, dt=float(dt), final_time=0.1)
    time_errors.append(l2_error(result))

plt.figure(figsize=(7, 4))
plt.loglog(time_steps, time_errors, 'o-', label='gemessener Fehler')
reference = time_errors[-1] * (time_steps / time_steps[-1]) ** 2
plt.loglog(time_steps, reference, '--', label=r'Referenz $O(\Delta t^2)$')
plt.xlabel('Zeitschritt Δt')
plt.ylabel('L2-Fehler')
plt.title('Zeitliche Konvergenz')
plt.grid(which='both', alpha=0.3)
plt.legend()
plt.show()


## Fazit

Das Beispiel zeigt die typische Arbeitsweise der numerischen Mathematik: Modell formulieren, Variationsproblem aufstellen, Raum und Zeit diskretisieren und die Näherung systematisch gegen eine bekannte Lösung prüfen.